# Step 2: LoRA models

Trains the 12 LoRA models (2 datasets x 2 backbones x 3 seeds). Uses the same Drive folder and `GH_TOKEN` as notebook 1.

1. Runtime > Change runtime type > T4 GPU.
2. Run all, and keep the tab open.
3. If Colab disconnects, just Run all again. Finished seeds get skipped and the unfinished one picks up from its last epoch.

After any git push, reset the runtime first or this keeps running the old code.

In [ ]:
import os
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "no gpu, change the runtime type to T4"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/code
!pip install -q -r requirements.txt

# outputs go to drive so they survive a disconnect
os.environ["GRSATS_OUT"] = "/content/drive/MyDrive/grsats/outputs"

In [ ]:
# unzip to the colab disk, reading from drive is really slow
ham = "../data/HAM10000/dataverse_files"
!mkdir -p {ham}
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# counts should match the paper and missing image files should be 0
!python check_data.py
!python -m pytest -q

In [ ]:
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python train_lora.py --dataset {ds} --backbone {bb}

In [ ]:
# linear probe vs lora on the clean pool, mean over the 3 seeds
import pandas as pd
from pathlib import Path

out = Path(os.environ["GRSATS_OUT"])
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        lp = pd.read_csv(out / f"{ds}_{bb}_lp_results.csv").groupby("group", sort=False)[["auc", "nll"]].mean()
        lora = pd.read_csv(out / f"{ds}_{bb}_lora_results.csv").groupby("group", sort=False)[["auc", "nll"]].mean()
        print(f"\n{ds} {bb}")
        print(lp.join(lora, lsuffix="_lp", rsuffix="_lora").round(3))